# DeepHP Dataset: 33 Experiments vs 19 WSIs Investigation

## Discrepancy
- Paper claims: **19 clinical whole-slide images (WSIs)** from Hospital Universitário João de Barros Barreto in Belém, Brazil
- Our implementation: **33 experiments** in CONFIG 87771 stratification

**Question**: Are these 33 experiments derived from the 19 WSIs? Are multiple patches/regions extracted from each WSI?

## Step 1: Extract All Experiment IDs from CONFIG 87771

In [ ]:
# Hardcoded experiment assignments from CONFIG 87771 in dataset_deepHP.py
config_87771_experiments = {
    0: [
        'Experiment-679',
        'Lm_449818_20x_13_03_2019',
        'Experiment-677',
        'Experiment-88',
        'Experiment-6716',
        'Experiment-68',
        'Experiment-671',
    ],
    1: [
        'Experiment-678',
        'Experiment-6712',
        'Experiment-673',
        'Experiment-101',
        'Experiment-6710',
        'Experiment-6717',
        'Experiment-676',
        'Experiment-672',
        'Experiment-99',
        'Experiment-6713',
    ],
    2: [
        'Experiment-97',
        'Experiment-674',
        'Lm_456061_20x_25_04_2019',
        'Experiment-675',
        'Experiment-91',
    ],
    3: [
        'Experiment-6711',
        'Experiment-108',
        'Experiment-105',
        'Lm_462218_20x_14_03_2019',
    ],
    4: [
        'Experiment-67',
        'Experiment-100',
        'Experiment-102',
        'Experiment-93',
        'Snap-151',
        'Experiment-6715',
        'Experiment-6714',
    ]
}

# Flatten all experiments
all_experiments = []
for fold, exps in config_87771_experiments.items():
    all_experiments.extend(exps)

print(f"Total experiments in CONFIG 87771: {len(all_experiments)}")
print(f"\nExperiments by prefix:")

import re
prefixes = {}
for exp in all_experiments:
    if exp.startswith('Experiment-'):
        prefix = 'Experiment'
    elif exp.startswith('Lm_'):
        prefix = 'Lm (microscope ID)'
    elif exp.startswith('Snap-'):
        prefix = 'Snap'
    else:
        prefix = 'Other'
    
    if prefix not in prefixes:
        prefixes[prefix] = []
    prefixes[prefix].append(exp)

for prefix, exps in sorted(prefixes.items()):
    print(f"  {prefix}: {len(exps)} experiments")
    print(f"    {exps}")

## Step 2: Analyze Experiment ID Patterns

Look for patterns in the numbering that might indicate which WSI they come from.

In [ ]:
# Extract numeric portions
experiment_numbers = {}

for exp in all_experiments:
    if exp.startswith('Experiment-'):
        num_str = exp.replace('Experiment-', '')
        try:
            num = int(num_str)
            experiment_numbers[exp] = num
        except ValueError:
            print(f"Could not parse: {exp}")

print(f"Experiment numbers extracted: {len(experiment_numbers)}")
print(f"\nExperiment-XXX IDs sorted:")
sorted_nums = sorted(experiment_numbers.items(), key=lambda x: x[1])
for exp, num in sorted_nums:
    print(f"  {exp}: {num}")

## Step 3: Check DeepHP Dataset Directory for Clues

List the actual directory structure to see if the experiments map to WSIs.

In [ ]:
import os
from pathlib import Path
from config import DEEPHP_DATASET_ROOT

print(f"DeepHP dataset root: {DEEPHP_DATASET_ROOT}")
print(f"Exists: {os.path.exists(DEEPHP_DATASET_ROOT)}")

if os.path.exists(DEEPHP_DATASET_ROOT):
    print(f"\nDirectory contents:")
    for item in sorted(os.listdir(DEEPHP_DATASET_ROOT)):
        item_path = os.path.join(DEEPHP_DATASET_ROOT, item)
        if os.path.isdir(item_path):
            num_files = len([f for f in os.listdir(item_path) if f.endswith('.png') or f.endswith('.jpeg') or f.endswith('.jpg')])
            print(f"  {item}/ ({num_files} image files)")
        else:
            print(f"  {item} (file)")

## Step 4: Sample Filenames to Identify WSI Mapping

Check actual filenames to understand the experiment-to-WSI mapping.

In [ ]:
# Sample filenames from Positive folder
positive_dir = os.path.join(DEEPHP_DATASET_ROOT, 'Positive')
if os.path.exists(positive_dir):
    files = os.listdir(positive_dir)[:10]  # First 10 files
    print("Sample filenames from Positive/:")
    for f in files:
        print(f"  {f}")
        
    print(f"\nExtract experiment IDs from filenames:")
    experiment_ids_found = set()
    for f in os.listdir(positive_dir):
        # Extract experiment ID: "Experiment-67_b0s0c0..." -> "Experiment-67"
        exp_id = f.split('_b0s')[0] if '_b0s' in f else f.split('_')[0]
        experiment_ids_found.add(exp_id)
    
    print(f"\nUnique experiment IDs in Positive/: {len(experiment_ids_found)}")
    for exp_id in sorted(experiment_ids_found):
        print(f"  {exp_id}")

## Step 5: Hypothesis - Multiple Regions per WSI

The paper mentions 19 WSIs from the hospital. Our implementation has 33 experiments.

**Possibility 1**: Each WSI has multiple regions/experiments extracted
- 33 experiments / 19 WSIs ≈ 1.7 experiments per WSI on average
- Some WSIs might have 1-2 regions, others might have more

**Possibility 2**: The naming convention (Experiment-XXX) refers to processing experiments, not WSI sources
- Multiple processing experiments per WSI
- Or different preprocessing/annotation protocols

**Possibility 3**: DeepHP paper describes 19 "clinical" WSIs but there are additional scans/repetitions

Let's check the filename patterns more carefully:

In [ ]:
# Analyze filename structure in detail
positive_dir = os.path.join(DEEPHP_DATASET_ROOT, 'Positive')
negative_dir = os.path.join(DEEPHP_DATASET_ROOT, 'Negative')

filename_patterns = {}

for directory in [positive_dir, negative_dir]:
    if os.path.exists(directory):
        for filename in os.listdir(directory):
            # Parse filename structure
            # Example: "Experiment-67_b0s0c0x10241280y10241280m65_0256x0256.jpeg"
            parts = filename.split('_')
            if len(parts) >= 3:
                exp_prefix = parts[0]  # "Experiment-67" or "Lm_449818..." or "Snap-151"
                # b0s0c0 = batch 0, series 0, channel 0
                batch_series = parts[1] if len(parts) > 1 else ""
                # x10241280y... = coordinates
                coords = parts[2] if len(parts) > 2 else ""
                # m65 = magnification 65x
                # 0256x0256 = patch size
                
                if exp_prefix not in filename_patterns:
                    filename_patterns[exp_prefix] = {'batch_series': set(), 'magnifications': set()}
                
                if batch_series:
                    filename_patterns[exp_prefix]['batch_series'].add(batch_series)
                
                # Extract magnification
                if 'm' in coords:
                    mag = coord.split('m')[1].split('_')[0]
                    filename_patterns[exp_prefix]['magnifications'].add(f"m{mag}")

print("Filename pattern analysis:")
print(f"Found {len(filename_patterns)} unique experiment prefixes\n")

for exp_id in sorted(filename_patterns.keys()):
    patterns = filename_patterns[exp_id]
    print(f"{exp_id}:")
    print(f"  Batch/Series combinations: {len(patterns['batch_series'])} (e.g., {list(patterns['batch_series'])[:3]})")
    print(f"  Magnifications: {list(patterns['magnifications'])}")

## Step 6: Research Question Resolution

Based on the analysis above, determine:
1. Are the 33 "experiments" all from 19 WSIs?
2. What does "Experiment-XXX" actually refer to?
3. Are multiple region/experiments extracted from each WSI?
4. What's the relationship between the Experiment IDs and the original hospital slides?

In [ ]:
# Summary of findings
print("=" * 70)
print("INVESTIGATION SUMMARY")
print("=" * 70)

print("\n📋 WHAT WE KNOW FROM THE PAPER:")
print("  - DeepHP dataset: 19 clinical WSIs from Hospital João de Barros Barreto")
print("  - Location: Belém, Brazil")
print("  - Total patches: 394,926 (120,374 positive, 274,551 negative)")

print("\n📋 WHAT OUR IMPLEMENTATION HAS:")
print(f"  - CONFIG 87771: 33 'experiments' in hardcoded fold assignments")
print(f"  - Experiment types:")
for prefix, exps in sorted(prefixes.items()):
    print(f"    * {prefix}: {len(exps)} experiments")

print("\n❓ DISCREPANCY:")
print(f"  - Paper: 19 WSIs")
print(f"  - Our code: 33 experiments")
print(f"  - Ratio: 33/19 ≈ 1.74 experiments per WSI")

print("\n🔍 LIKELY EXPLANATION:")
print("  Each of the 19 clinical WSIs has been divided into MULTIPLE REGIONS/EXPERIMENTS")
print("  during preprocessing. Each experiment represents a distinct scanning region,")
print("  magnification level, or processing batch from the same original WSI.")

print("\n⚠️  IMPLICATION FOR STRATIFICATION:")
print("  Our CONFIG 87771 assigns experiments (not WSIs) to folds.")
print("  This means:")
print("  - ✅ Good: No patch-level leakage (patches from same experiment stay together)")
print("  - ⚠️  Potential issue: Regions from same WSI might split across train/val")
print("     if they're treated as separate experiments")
print("  - 🔬 Need to verify: Does the paper recommend WSI-level or experiment-level CV?")

## Next Steps

To fully resolve this discrepancy, we need to:

1. **Find the DeepHP paper** and check:
   - How they define "19 WSIs" - are these the original slides?
   - Do they mention preprocessing that creates multiple experiments per WSI?
   - What stratification strategy do they use (WSI-level or patch-level)?

2. **Map Experiment IDs to WSIs**:
   - Check if there's metadata file that maps Experiment-XXX to original WSI IDs
   - Look for naming patterns that indicate parent WSI

3. **Validate our stratification**:
   - If experiments map to the same WSI should they stay together?
   - Or is experiment-level separation acceptable for avoiding artifact learning?

4. **Check related papers or DeepHP publication** for dataset details